In [0]:
%sql

-- Checking customer.csv table
select * from csv.`/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv`;

In [0]:
%sql

-- Checking order.csv table
select * from csv.`/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/orders.csv`;

In [0]:
%sql

-- Checking product.csv table
select * from csv.`/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/products.csv`;

In [0]:
%sql

-- table creted in broze
CREATE TABLE IF NOT EXISTS databricks_incremental_medallion_lakehouse.bronze.customers_bronze (
customer_id integer,
customer_name string,
city string,
state string, 
updated_at timestamp
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_incremental_medallion_lakehouse.bronze.orders_bronze (
 order_id integer ,
 customer_id integer ,
 amount double ,
 order_date date ,
 status string ,
 updated_at timestamp )
 USING  DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_incremental_medallion_lakehouse.bronze.products_bronze (
 product_id integer,
 product_name string, 
 category string, 
 price double, 
 updated_at timestamp) 
 USING  DELTA;

In [0]:
customers_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv", header=True, inferSchema=True)

In [0]:
customers_df.display()

In [0]:
orders_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/orders.csv", header=True, inferSchema=True)

In [0]:
orders_df.display()

In [0]:
products_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/products.csv", header=True, inferSchema=True)

In [0]:
products_df.display()

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
#Add ingestion timestamp
customers_df = customers_df.withColumn(
    "ingestion_date",
    F.current_timestamp()
)


In [0]:
customers_df.display()

In [0]:
orders_df = orders_df.withColumn(
    "ingestion_date",
    F.current_timestamp()
)

In [0]:
orders_df.display()

In [0]:
products_df = products_df.withColumn(
    "ingestion_date",
    F.current_timestamp())

In [0]:
products_df.display()

In [0]:
%sql
select * from csv.`/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv`

In [0]:
#Convert updated_at to timestamp
customers_df = customers_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at"))



In [0]:
orders_df = orders_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at"))


In [0]:
products_df = products_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at"))

In [0]:
customers_df.printSchema()

In [0]:
orders_df.printSchema()

In [0]:
products_df.printSchema()

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.customers_bronze;

In [0]:
%sql
Alter table databricks_incremental_medallion_lakehouse.bronze.customers_bronze
add columns (ingestion_date TIMESTAMP,
initial_load_date TIMESTAMP,
last_updated_date TIMESTAMP
);

In [0]:
%sql
Alter table databricks_incremental_medallion_lakehouse.bronze.orders_bronze
add columns (ingestion_date TIMESTAMP,
initial_load_date TIMESTAMP,
last_updated_date TIMESTAMP
);

In [0]:
%sql
Alter table databricks_incremental_medallion_lakehouse.bronze.products_bronze
add columns (ingestion_date TIMESTAMP,
initial_load_date TIMESTAMP,
last_updated_date TIMESTAMP
);

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.orders_bronze

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.products_bronze

In [0]:
customers_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv", header=True, inferSchema=True)

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.customers_bronze")

In [0]:
orders_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/orders.csv", header=True, inferSchema=True)

In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.orders_bronze")

In [0]:
products_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/products.csv", header=True, inferSchema=True)

In [0]:
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_incremental_medallion_lakehouse.bronze.products_bronze")

In [0]:
%sql
select * from databricks_incremental_medallion_lakehouse.bronze.customers_bronze;


In [0]:
customers_df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv", header=True, inferSchema=True)

In [0]:
customers_df.display()

In [0]:
from pyspark.sql.functions import when, col, current_timestamp

customers_df = customers_df.withColumn(
    "customer_name",
    when(col("customer_id") == 102, "David Khan")
    .otherwise(col("customer_name"))
).withColumn(
    "city",
    when(col("customer_id") == 102, "Pune")
    .otherwise(col("city"))
).withColumn(
    "updated_at",
    when(col("customer_id") == 102, current_timestamp())
    .otherwise(col("updated_at"))
)


In [0]:
customers_df.display()

In [0]:
df = spark.read.csv("/Volumes/databricks_incremental_medallion_lakehouse/default/incremental_load_files/customers.csv", header=True, inferSchema=True)

In [0]:
df.display()

###Create Audit table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_incremental_medallion_lakehouse.     bronze.pipeline_audit
(
    pipeline_name STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA;